In [1]:
import scipy
from PIL import Image
import matplotlib.pyplot as plt 
import numpy as np
from scipy.signal import butter, filtfilt
from scipy.fftpack import fft2, fftshift
from tqdm import tqdm
import os

In [ ]:
IMAGE_DIR = 'FaceForensics++/manipulated_sequences/Deepfakes/c23/frames/000_003' 
N_IMAGES = 100
PATCHES_PER_IMAGE = 50
IMAGE_SIZE = 256
PATCH_SIZE = 32  


cutoff = 10
fs = 100
order = 2
btype = 'low' 

#  Design Butterworth Filter 
b, a = butter(N=order, Wn=cutoff / (0.5 * fs), btype=btype)

# Load image file paths 
image_files = sorted([
    f for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
])[:N_IMAGES]

spectra = []

for fname in tqdm(image_files, desc="Processing images"):
    path = os.path.join(IMAGE_DIR, fname)

    #  Load and resize image 
    try:
        img = Image.open(path).convert('L')
        img = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.LANCZOS)
        img_np = np.asarray(img, dtype=np.float32) / 255.0
    except Exception as e:
        print(f"Skipping {fname}: {e}")
        continue

    #  Extract non-overlapping patches 
    patches = []
    for y in range(0, IMAGE_SIZE, PATCH_SIZE):
        for x in range(0, IMAGE_SIZE, PATCH_SIZE):
            patch = img_np[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if patch.shape == (PATCH_SIZE, PATCH_SIZE):
                patches.append(patch)
            if len(patches) == PATCHES_PER_IMAGE:
                break
        if len(patches) == PATCHES_PER_IMAGE:
            break

    #  Process each patch 
    for patch in patches:
        lowpass = np.zeros_like(patch)
        for i in range(patch.shape[0]):
            lowpass[i, :] = filtfilt(b, a, patch[i, :])
        for j in range(patch.shape[1]):
            lowpass[:, j] = filtfilt(b, a, lowpass[:, j])

        highpass = patch - lowpass

        # FFT and log spectrum
        f = np.fft.fft2(highpass)
        fshift = np.fft.fftshift(f)
        magnitude = np.log(1 + np.abs(fshift))

        spectra.append(magnitude)

#  Average log-magnitude spectrum of all patches 
spectra = np.array(spectra)
avg_spectrum = np.mean(spectra, axis=0)

plt.figure(figsize=(6, 6))
plt.imshow(avg_spectrum, cmap='gray')
plt.title("Average Log-Magnitude Spectrum ")
plt.axis('off')
plt.tight_layout()
output_path = 'output/FF-spectrum.png'  
plt.savefig(output_path, dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

Processing images:  50%|█████     | 16/32 [00:07<00:07,  2.06it/s]